# The Price is Right

## Week 8 Order of Play

Day 1: Modal.com and SpecialistAgent  
Day 2: RAG, FrontierAgent, Ensemble Agent  
Day 3: ScannerAgent, MessengerAgent  
Day 4: AutonomousPlannerAgent and DealAgentFramework  
Day 5: The Price Is Right Finale


Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [1]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection
import logging
import requests
load_dotenv(override=True)
openai = OpenAI()
MODEL = 'gpt-5-mini'

In [2]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 3/3 [01:02<00:00, 20.84s/it]


In [3]:
len(deals)

30

In [4]:
deals[10].describe()

'Title: Projector Deals at Best Buy: Up to 44% off + free shipping\nDetails: Save up to 44% off home theater projectors in this selection. Shop deals on 4K and 1080p models with Google TV, Roku built-in, and Dolby Audio, with prices starting at $146. One standout is the Aurzen Smart Projector with Built-in Google TV for $250 (was $450), a $200 discount. Shop Now at Best Buy\nFeatures: \nURL: https://www.dealnews.com/Projector-Deals-at-Best-Buy-Up-to-44-off-free-shipping/21811114.html?iref=rss-c39'

### We are going to ask GPT-5-mini to summarize deals and identify their price

In [5]:
SYSTEM_PROMPT = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 
"""

USER_PROMPT_PREFIX = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""

USER_PROMPT_SUFFIX = "\n\nInclude exactly 5 deals, no more."

In [6]:
# this makes a suitable user prompt given scraped deals

def make_user_prompt(scraped):
    user_prompt = USER_PROMPT_PREFIX
    user_prompt += '\n\n'.join([scrape.describe() for scrape in scraped])
    user_prompt += USER_PROMPT_SUFFIX
    return user_prompt

In [7]:
# Let's create a user prompt for the deals we just scraped, and look at how it begins

user_prompt = make_user_prompt(deals)
print(user_prompt[:2000])
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user_prompt}]

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price that is greater than 0.
You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a short paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Best Buy Clearance Deals of the Week: Up to 60% off + free shipping
Details: Best Buy is hosting its Clearance Days sale with discounts of up to 60% off a wide array of tech and home essentials. The event includes deals on TVs and home theater gear, laptops and tablets, smart home devices, appliances, headphones, and gaming gear. Shop Now at Best Buy
Features: 
URL: https://www

In [8]:
response = openai.chat.completions.parse(model=MODEL, messages=messages, response_format=DealSelection, reasoning_effort="minimal")
results = response.choices[0].message.parsed
results

DealSelection(deals=[Deal(product_description='Fremo TP300 is a 300W portable power station with a 100W USB-C output suitable for charging laptops, phones, and powering small appliances. It’s a compact, transportable battery pack designed for camping, emergencies, and mobile work, offering multiple output ports to support a variety of devices. The unit emphasizes practical portable power in a small form factor with enough capacity for short-term off-grid use.', price=130.0, url='https://www.dealnews.com/products/Fremo/Fremo-TP300-300-W-Portable-Power-Station/459756.html?iref=rss-c142'), Deal(product_description='Onn model 100012589 is a 32-inch Roku Smart TV with a native 1280x720 (720p) LED panel and three HDMI inputs. It runs Roku’s smart TV platform for streaming apps and simple navigation, and its compact size makes it suitable for bedrooms, kitchens, or secondary rooms where full 1080p/4K resolution isn’t required. The open-box unit is described as in excellent condition and inclu

In [9]:
for deal in results.deals:
    print(deal.product_description)
    print(deal.price)
    print(deal.url)
    print()


Fremo TP300 is a 300W portable power station with a 100W USB-C output suitable for charging laptops, phones, and powering small appliances. It’s a compact, transportable battery pack designed for camping, emergencies, and mobile work, offering multiple output ports to support a variety of devices. The unit emphasizes practical portable power in a small form factor with enough capacity for short-term off-grid use.
130.0
https://www.dealnews.com/products/Fremo/Fremo-TP300-300-W-Portable-Power-Station/459756.html?iref=rss-c142

Onn model 100012589 is a 32-inch Roku Smart TV with a native 1280x720 (720p) LED panel and three HDMI inputs. It runs Roku’s smart TV platform for streaming apps and simple navigation, and its compact size makes it suitable for bedrooms, kitchens, or secondary rooms where full 1080p/4K resolution isn’t required. The open-box unit is described as in excellent condition and includes necessary accessories.
69.0
https://www.dealnews.com/products/Onn/Onn-100012589-32-72

In [10]:
root = logging.getLogger()
root.setLevel(logging.INFO)

In [11]:
from agents.scanner_agent import ScannerAgent

In [12]:
agent = ScannerAgent()
result = agent.scan()

INFO:root:[Scanner Agent] Scanner Agent is initializing
INFO:root:[Scanner Agent] Scanner Agent is ready
INFO:root:[Scanner Agent] Scanner Agent is about to fetch deals from RSS feed
INFO:root:[Scanner Agent] Scanner Agent received 30 deals not already scraped
INFO:root:[Scanner Agent] Scanner Agent is calling OpenAI using Structured Outputs
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:[Scanner Agent] Scanner Agent received 5 selected deals with price>0 from OpenAI


In [13]:
result

DealSelection(deals=[Deal(product_description='Fremo TP300 is a 300-watt portable power station designed for charging small to medium devices on the go. It includes a 100W USB-C port suitable for fast-charging laptops and modern devices, multiple output options for AC and USB accessories, and a compact form factor intended for camping, emergencies, or mobile work. The unit targets users who need reliable portable power without the bulk of larger inverter generators.', price=130.0, url='https://www.dealnews.com/products/Fremo/Fremo-TP300-300-W-Portable-Power-Station/459756.html?iref=rss-c142'), Deal(product_description='Onn model 100012589 is a 32-inch 720p LED Roku Smart TV offering a budget-friendly smart TV experience. It features a 1280x720 native resolution, Roku OS with streaming apps and channel store, and three HDMI inputs for connecting consoles, streaming sticks, or set-top boxes. The TV is positioned as an entry-level living room or bedroom display with essential smart featur

### Introducing Pushover

Pushover is a nifty tool for sending Push Notifications to your phone.

It's super easy to set up and install!

Simply visit https://pushover.net/ and click 'Login or Signup' on the top right to sign up for a free account, and create your API keys.

Once you've signed up, on the home screen, click "Create an Application/API Token", and give it any name (like AIEngineer) and click Create Application.

Then add 2 lines to your `.env` file:

PUSHOVER_USER=_put the key that's on the top right of your Pushover home screen and probably starts with a u_  
PUSHOVER_TOKEN=_put the key when you click into your new application called Agents (or whatever) and probably starts with an a_

Remember to save your `.env` file, and run `load_dotenv(override=True)` after saving, to set your environment variables.

Finally, click "Add Phone, Tablet or Desktop" to install on your phone.

In [14]:
load_dotenv(override=True)

True

In [15]:
pushover_user = os.getenv('PUSHOVER_USER')
pushover_token = os.getenv('PUSHOVER_TOKEN')
pushover_url = "https://api.pushover.net/1/messages.json"

In [16]:
if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

Pushover user found and starts with u
Pushover token found and starts with a


In [17]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [18]:
push("MASSIVE DEAL!!")

Push: MASSIVE DEAL!!


In [19]:
from agents.messaging_agent import MessagingAgent

agent = MessagingAgent()
agent.push("SUCH A MASSIVE DEAL!!")

INFO:root:[Messaging Agent] Messaging Agent is initializing
INFO:root:[Messaging Agent] Messaging Agent has initialized Pushover and openai
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification


In [20]:
agent.notify("A special deal on Sumsung 60 inch LED TV going at a great bargain", 300, 1000, "www.samsung.com")

INFO:root:[Messaging Agent] Messaging Agent is using gpt-5-nano to craft the message
INFO:root:[Messaging Agent] Messaging Agent is sending a push notification
INFO:root:[Messaging Agent] Messaging Agent has completed
